# Chess: Predict Black Moves from White Moves

The task consist in guessing the sequence of Black Moves in a chess game, given the sequence of White Moves. The performace is measured in terms of the longest correct prefix.

Each game is represented by two aligned sequences:

```text
white: e4 Nf3 Bb5 Ba4 O-O
black: e5 Nc6 a6 Nf6 Be7
```

The input is the sequence of White moves.  
The target is the corresponding sequence of Black moves.

The dataset consists of CSV files with two columns:

```text
white,black
```

The baseline model is a simple LSTM:

```text
White moves → Embedding → LSTM → Black move at each timestep
```


## 1. Setup

In [1]:
import os
import numpy as np
import pandas as pd
import gdown
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers
from tqdm.auto import tqdm

print("TensorFlow:", tf.__version__)


TensorFlow: 2.20.0


# Data dowloading

we recommend to create local copies

In [2]:
!gdown 1z6ik6MyTjZLVvdsCKTFAgsL6uiSbeKs6
!gdown 1aQF9QQO0GgNLgv13bf5BfExaVij2OowX

Downloading...
From: https://drive.google.com/uc?id=1z6ik6MyTjZLVvdsCKTFAgsL6uiSbeKs6
To: /content/white_to_black_test.csv
100% 4.89M/4.89M [00:00<00:00, 28.5MB/s]
Downloading...
From: https://drive.google.com/uc?id=1aQF9QQO0GgNLgv13bf5BfExaVij2OowX
To: /content/white_to_black_train.csv
100% 27.8M/27.8M [00:00<00:00, 55.2MB/s]


## 2. Load train and validation data

You might want to split the train set into train and validation.

In [3]:
TRAIN_PATH = "white_to_black_train.csv"
VALID_PATH = "white_to_black_test.csv"

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(VALID_PATH)

assert set(train_df.columns) == {"white", "black"}
assert set(test_df.columns) == {"white", "black"}

train_df = train_df.dropna().astype(str)
test_df = test_df.dropna().astype(str)

train_df["white_len"] = train_df["white"].str.split().str.len()
train_df["black_len"] = train_df["black"].str.split().str.len()
test_df["white_len"] = test_df["white"].str.split().str.len()
test_df["black_len"] = test_df["black"].str.split().str.len()

train_df = train_df[train_df["white_len"] == train_df["black_len"]].copy()
test_df = test_df[test_df["white_len"] == test_df["black_len"]].copy()

print("Train:", train_df.shape)
print("Valid:", test_df.shape)

train_df.head()


Train: (101364, 4)
Valid: (17888, 4)


,white,black,white_len,black_len
0,e4 d4 Bd3 Qe2 c4 Nc3 h4 Bf4 Qd2 Nf3 Rh3 Bxh6 R...,b6 Bb7 Nf6 e6 Be7 O-O h6 d6 Ng4 Nd7 e5 Nxh6 Bx...,34,34
1,e4 d4 e5 Nf3 c3 a3 b3 cxd4 Bd2 Bc3 Be2 Nxe5 O-...,c6 e6 d5 c5 Nc6 Qb6 cxd4 Qa5+ Qb6 f6 fxe5 Bd6 ...,19,19
2,c4 b4 cxd5 d4 Ba3 e3 Bb2 Nf3 Ne5 Nxd7 Bb5 O-O ...,c6 d5 cxd5 a5 e6 axb4 Bd7 Bd6 Nc6 Qxd7 Nge7 O-...,17,17
3,e4 Nf3 d4 c3 e5 Bb5 Qa4 Bxc6+ gxf3 e6+ Qxc6+ c...,Nc6 e5 exd4 d5 Bg4 f6 Bxf3 Kf7 bxc6 Kxe6 Kf7 B...,29,29
4,e4 f3 d3 e5 f4 exf6 c3 b4 Nh3 g4 gxf5 Ng5 Ne6 ...,Nf6 d5 e6 Nfd7 f6 Bb4+ Ba5 Bb6 gxf6 f5 exf5 h6...,59,59


The training set contains approximately 100K games. **You are not allowed to extend it with additional games.**

## 3. Vectorization

In [ ]:
MAX_LEN = 60
WHITE_VOCAB_SIZE = 20000
BLACK_VOCAB_SIZE = 20000

def identity_standardize(x):
    return x

white_vectorizer = layers.TextVectorization(
    max_tokens=WHITE_VOCAB_SIZE,
    standardize=identity_standardize,
    split="whitespace",
    output_mode="int",
    output_sequence_length=MAX_LEN,
)

black_vectorizer = layers.TextVectorization(
    max_tokens=BLACK_VOCAB_SIZE,
    standardize=identity_standardize,
    split="whitespace",
    output_mode="int",
    output_sequence_length=MAX_LEN,
)

white_vectorizer.adapt(train_df["white"].values)
black_vectorizer.adapt(train_df["black"].values)

white_vocab = white_vectorizer.get_vocabulary()
black_vocab = black_vectorizer.get_vocabulary()

print("White vocabulary:", len(white_vocab))
print("Black vocabulary:", len(black_vocab))
print("White examples:", white_vocab[:20])
print("Black examples:", black_vocab[:20])


White vocabulary: 4363
Black vocabulary: 5115
White examples: ['', '[UNK]', np.str_('Nf3'), np.str_('e4'), np.str_('d4'), np.str_('Nc3'), np.str_('O-O'), np.str_('c4'), np.str_('h3'), np.str_('c3'), np.str_('f4'), np.str_('Bc4'), np.str_('a3'), np.str_('d3'), np.str_('Bd3'), np.str_('g3'), np.str_('b4'), np.str_('Be3'), np.str_('Bg5'), np.str_('Re1')]
Black examples: ['', '[UNK]', np.str_('Nf6'), np.str_('Nc6'), np.str_('d5'), np.str_('e5'), np.str_('O-O'), np.str_('e6'), np.str_('c5'), np.str_('d6'), np.str_('Be7'), np.str_('c6'), np.str_('h6'), np.str_('g6'), np.str_('a6'), np.str_('b5'), np.str_('f5'), np.str_('b6'), np.str_('f6'), np.str_('a5')]


## 4. TensorFlow datasets

In [ ]:
BATCH_SIZE = 64

def make_dataset(dataframe, shuffle=True):
    white = dataframe["white"].values
    black = dataframe["black"].values

    ds = tf.data.Dataset.from_tensor_slices((white, black))

    if shuffle:
        ds = ds.shuffle(buffer_size=min(len(dataframe), 10000), seed=42)

    def vectorize_batch(white_text, black_text):
        x = white_vectorizer(white_text)
        y = black_vectorizer(black_text)
        return x, y

    return ds.batch(BATCH_SIZE).map(vectorize_batch).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(train_df, shuffle=True)
test_ds = make_dataset(test_df, shuffle=False)

x_batch, y_batch = next(iter(train_ds))
x_batch.shape, y_batch.shape


(TensorShape([64, 60]), TensorShape([64, 60]))

# IMPORTANT

1. If you think it could be beneficial, during training you can use, as input data, both white and black moves (e.g. in a teacher-forcing approach).  Obviously, during testing ou can only see white moves. That means that, if you need black moves, generation must be done in autoregressive way, using your own predictions as input.

2. In order to speed up training and save space you might tokenize data in a preprocessing phase. Please do the final evaluation on decoded data, though, for ease of inspection.


## Prediction utility

In [ ]:
black_vocab = black_vectorizer.get_vocabulary()
black_id_to_token = {i: token for i, token in enumerate(black_vocab)}

def predict_black_sequence(white_text):
    x = white_vectorizer(tf.constant([white_text]))
    probs = model(x, training=False).numpy()  #use your model here
    pred_ids = np.argmax(probs[0], axis=-1)

    n = min(len(white_text.split()), MAX_LEN)
    tokens = []

    for idx in pred_ids[:n]:
        token = black_id_to_token.get(int(idx), "")
        if token == "":
            continue
        tokens.append(token)

    return tokens

#example = valid_df.iloc[0]
#print("WHITE:", example["white"])
#print("TRUE BLACK:", example["black"])
#print("PRED BLACK:", " ".join(predict_black_sequence(example["white"])))

## Project Metric: Survival AUC

For each game, define the **correct prefix length** as the number of consecutive Black moves correctly predicted before the first error.

Example:

```text
true: e5 Nc6 a6 Nf6 Be7
pred: e5 Nc6 a6 d6  Be7
```

The first error occurs at move 4, so the correct prefix length is 3.

The survival curve is:

```text
S(k) = fraction of games with correct prefix length at least k
```

The Survival AUC is:

```text
AUC = sum_{k >= 1} S(k)
```

With this convention, the AUC equals the mean correct prefix length.


In [ ]:
def prefix_length(true_tokens, pred_tokens):

    n = min(len(true_tokens), len(pred_tokens))
    count = 0
    for i in range(n):
        if true_tokens[i] != pred_tokens[i]:
            break
        count += 1

    return count


def evaluate_survival_auc(dataframe, max_examples=1000):
    records = []
    total_correct = 0
    total_moves = 0

    subset = dataframe.head(max_examples)

    for _, row in tqdm(subset.iterrows(), total=len(subset)):
        true_tokens = row["black"].split()[:MAX_LEN]
        pred_tokens = predict_black_sequence(row["white"])

        n = min(len(true_tokens), len(pred_tokens))

        for i in range(len(true_tokens)):
            if i < n and true_tokens[i] == pred_tokens[i]:
                total_correct += 1
            total_moves += 1

        prefix = prefix_length(true_tokens, pred_tokens)

        records.append({
            "white": row["white"],
            "true_black": row["black"],
            "pred_black": " ".join(pred_tokens),
            "length": len(true_tokens),
            "correct_prefix": prefix
        })

    results = pd.DataFrame(records)

    move_accuracy = total_correct / total_moves
    mean_prefix = results["correct_prefix"].mean()

    max_prefix = int(results["correct_prefix"].max())

    survival = np.array([
        (results["correct_prefix"] >= k).mean()
        for k in range(1, max_prefix + 1)
    ])

    survival_auc = survival.sum()

    return {
        "results": results,
        "move_accuracy": move_accuracy,
        "mean_prefix": mean_prefix,
        "survival": survival,
        "survival_auc": survival_auc,
    }


## Evaluation.

The metric must be computed over the full test set.

In [ ]:
metrics = evaluate_survival_auc(
    test_df,
    start_move=1,
    max_examples=1000,
)

print("Move accuracy:", metrics["move_accuracy"])
#print("Mean correct prefix:", metrics["mean_prefix"])
print("Survival AUC:", metrics["survival_auc"])


## Plot survival curves

The following utility may be used to plot survival curves.

In [ ]:
import matplotlib.pyplot as plt

def plot_survival(metrics, title):
    survival = metrics["survival"]

    plt.figure(figsize=(7, 4))
    plt.plot(range(1, len(survival) + 1), survival)
    plt.xlabel("Number of consecutive correct Black moves")
    plt.ylabel("Fraction of games surviving")
    plt.title(title)
    plt.grid(True)
    plt.show()

plot_survival(metrics, "Survival from move 1")


##Inspect errors

You might be interested to inspect errors.

In [ ]:
errors = metrics["results"][metrics["results"]["correct_prefix"] < metrics["results"]["length"]]
errors[["white", "true_black", "pred_black", "correct_prefix", "length"]].head(10)

# Additional Constraints

* Data augmentation is not permitted.
* The total number of trainable parameters must remain below 6 million. The number of parameters of the proposed solution must be reported explicitly.
* The model weights must be made available for download via gdown. Please verify that the provided link works correctly and that the weights can be loaded successfully.
* The solution must be implemented in Keras and must run on Google Colab. Submissions containing notebook execution errors will be penalized.
* The submission must be a single, well documented notebook file. Explain and motivate your methodological choices.
**Tar files will be rejected.**





The problem is intrinsically nondeterministic and quite challenging. Do your best.

Good work!